# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Notebook này được thiết kế hoàn chỉnh để chạy trực tiếp trên **Kaggle Notebook** (hoặc **Google Colab T4**).
Toàn bộ quy trình tuân thủ chính xác các yêu cầu trong `README.md`, `GUIDE.md` và `RUBRIC.md`.

## 0. Môi trường (Colab T4 / Kaggle)
Kiểm tra GPU và cài đặt các thư viện cần thiết.
Trên Kaggle, GPU thường là T4 x2 hoặc P100 (16GB VRAM). Chúng ta sẽ dùng 1 GPU (`cuda:0`).

In [ ]:
# Kiểm tra thông tin GPU
!nvidia-smi -L

# Nếu chạy trên Kaggle/Colab, tự động clone repo về thư mục làm việc để lấy scripts/ và requirements
import os
from pathlib import Path

if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")

if not os.path.exists("requirements-colab.txt") or not os.path.exists("scripts"):
    print("Đang clone repository từ GitHub...")
    !git clone https://github.com/nguyenquangdao2004-glitch/K4-Track-4-Day10-VisionLanguageModel.git /kaggle/working/repo
    !cp -rn /kaggle/working/repo/* /kaggle/working/ 2>/dev/null || true
    print("Đã lấy xong toàn bộ mã nguồn repo!")

# Cài đặt các gói phụ thuộc theo requirements-colab.txt
!pip -q install -r requirements-colab.txt

In [ ]:
import os
import sys
from pathlib import Path

if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")
    print("Đang chạy trên Kaggle Notebook:", os.getcwd())
else:
    print("Đang chạy tại thư mục:", os.getcwd())

# Cấu hình Kaggle Secrets (bắt toàn bộ Exception để không bị TimeoutException trên Kaggle)
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME') or os.environ.get('KAGGLE_USERNAME', '')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY') or os.environ.get('KAGGLE_KEY', '')
except Exception:
    pass

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_KEY")
except Exception:
    pass

# Tự động khôi phục checkpoints work/ và report/ từ Output của version trước nếu được gắn làm Input
import shutil
if Path("/kaggle/input").exists():
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        p = Path(root_dir)
        if p.name == "work" and not (Path("/kaggle/working/work/E1").exists()):
            dst = Path("/kaggle/working/work")
            dst.mkdir(parents=True, exist_ok=True)
            print(f"Phát hiện checkpoint từ version trước tại {p}, đang khôi phục...")
            shutil.copytree(p, dst, dirs_exist_ok=True)
            print("Checkpoint hiện có trong work/:", [d.name for d in dst.iterdir() if d.is_dir()])
        if p.name == "report" and not (Path("/kaggle/working/report/evidence").exists()):
            dst_rep = Path("/kaggle/working/report")
            dst_rep.mkdir(parents=True, exist_ok=True)
            print(f"Phát hiện report từ version trước tại {p}, đang khôi phục...")
            shutil.copytree(p, dst_rep, dirs_exist_ok=True)

print("Môi trường đã sẵn sàng!")

## 1. Dữ liệu: tải và chia split
Tải dữ liệu từ Kaggle và chia thành 3 split (`train`, `val`, `test`) theo phiên quay video (`session`) cố định.

In [ ]:
# Tự động phát hiện dataset trong /kaggle/input hoặc tải qua Kaggle API
kaggle_input_dir = None
if Path("/kaggle/input").exists():
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        p = Path(root_dir)
        if (p / "images").exists() and (p / "labels-YOLO").exists():
            kaggle_input_dir = p
            break

if kaggle_input_dir is not None:
    print(f"Phát hiện dataset tại: {kaggle_input_dir}, tiến hành chia split trực tiếp...")
    !python scripts/prepare_data.py --source "{kaggle_input_dir}"
else:
    print("Tải dữ liệu qua Kaggle API...")
    !python scripts/prepare_data.py --kaggle

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

from scripts.labcommon import read_jsonl, CLASSES

# Đọc dữ liệu splits
meta = json.loads(Path("data/splits/meta.json").read_text(encoding="utf-8"))
print("Dataset version:", meta["dataset_version"])
print("Thống kê counts:", json.dumps(meta["counts"], indent=2, ensure_ascii=False))

train_data = read_jsonl("data/splits/train.jsonl")
val_data = read_jsonl("data/splits/val.jsonl")
test_data = read_jsonl("data/splits/test.jsonl")

print(f"Số lượng ảnh: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

# Xác nhận không có session nào xuất hiện ở hai split
train_sessions = {x['session'] for x in train_data}
val_sessions = {x['session'] for x in val_data}
test_sessions = {x['session'] for x in test_data}
assert len(train_sessions & val_sessions) == 0, "Rò rỉ train và val!"
assert len(train_sessions & test_sessions) == 0, "Rò rỉ train và test!"
assert len(val_sessions & test_sessions) == 0, "Rò rỉ val và test!"
print("Xác nhận: Không có session nào bị rò rỉ giữa các split!")

# Visualize 6 ảnh mẫu kèm bounding box
color_map = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, sample in enumerate(train_data[:6]):
    img_path = Path("data") / sample["image_path"]
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size
    for obj in sample["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h
        lbl = obj["label"]
        col = color_map.get(lbl, "white")
        draw.rectangle([x1, y1, x2, y2], outline=col, width=3)
        draw.text((x1, max(0, y1 - 10)), lbl, fill=col)
    axes[i].imshow(img)
    axes[i].set_title(f"{sample['image_id']} ({len(sample['objects'])} boxes)")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000
Chuyển đổi toạ độ từ chuẩn YOLO `(xc, yc, w, h)` [0, 1] sang định dạng grounding JSON với toạ độ số nguyên trong thang `[0, 1000]`.
Sắp xếp các box theo thứ tự đọc `(y1, x1)` từ trên xuống dưới, trái qua phải để model học autoregressive sequence ổn định.

In [ ]:
def yolo_to_box(label, yolo):
    """Chuyển 1 object YOLO sang định dạng grounding thang 0-1000."""
    xc, yc, w, h = yolo
    x1 = round((xc - w / 2) * 1000)
    y1 = round((yc - h / 2) * 1000)
    x2 = round((xc + w / 2) * 1000)
    y2 = round((yc + h / 2) * 1000)
    # Kẹp vào khoảng [0, 1000]
    x1 = max(0, min(1000, x1))
    y1 = max(0, min(1000, y1))
    x2 = max(0, min(1000, x2))
    y2 = max(0, min(1000, y2))
    return {"label": label, "bbox_2d": [int(x1), int(y1), int(x2), int(y2)]}

def objects_to_answer(objects):
    """Chuyển danh sách objects sang chuỗi JSON rút gọn, sắp xếp theo (y1, x1)."""
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    # Sắp xếp theo thứ tự (y1, x1): trên -> dưới, trái -> phải
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

# Prompt grounding chuẩn cho thang 0-1000
PROMPT_NORM1000 = (
    'Định vị các hư hại mặt đường trong ảnh gồm: pothole, crack, manhole. '
    'Trả về một danh sách JSON dạng [{"label": "<tên_lớp>", "bbox_2d": [x1, y1, x2, y2]}], '
    'trong đó toạ độ là các số nguyên trên thang 0-1000 [x1, y1, x2, y2]. '
    'Chỉ trả về JSON thuần, không kèm bất kỳ giải thích nào khác.'
)

# Kiểm tra hàm chuyển đổi với một mẫu
sample_ans = objects_to_answer(train_data[0]["objects"])
print("Ví dụ answer JSON:", sample_ans)

## 3. E0: zero-shot
Đánh giá mô hình gốc `Qwen/Qwen3-VL-4B-Instruct` dạng 4-bit (NF4) mà chưa qua finetune.
Lưu ý quan trọng: `llm_int8_skip_modules=["visual", "lm_head"]` để giữ ViT và projector ở fp16.

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# Cấu hình lượng tử hoá QLoRA 4-bit NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "merger", "deepstack", "deepstack_merger_list", "lm_head"]
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

print("Đang nạp model Qwen3-VL-4B 4-bit...")
model_e0 = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
model_e0.eval()
print("Đã nạp model E0 thành công!")

In [ ]:
from tqdm import tqdm
from scripts.labcommon import write_jsonl

# Suy luận Zero-shot trên tập test (batch size 4)
os.makedirs("work", exist_ok=True)
test_records = read_jsonl("data/splits/test.jsonl")

e0_predictions = []
batch_size = 4

for i in tqdm(range(0, len(test_records), batch_size), desc="E0 Inference"):
    batch_records = test_records[i:i + batch_size]
    batch_images = [Image.open(Path("data") / r["image_path"]).convert("RGB") for r in batch_records]
    
    messages_batch = []
    for img in batch_images:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": PROMPT_NORM1000}
            ]
        }]
        messages_batch.append(messages)
    
    # Chuẩn bị inputs
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_batch]
    inputs = processor(
        text=texts,
        images=batch_images,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        out = model_e0.generate(**inputs, max_new_tokens=256, do_sample=False)
    
    # Chỉ giải mã phần model sinh mới
    prompt_len = inputs["input_ids"].shape[1]
    decoded = processor.tokenizer.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
    
    for r, raw_pred in zip(batch_records, decoded):
        e0_predictions.append({"image_id": r["image_id"], "raw": raw_pred.strip()})

# Ghi ra work/e0.jsonl
write_jsonl("work/e0.jsonl", e0_predictions)
print(f"Đã ghi {len(e0_predictions)} kết quả vào work/e0.jsonl")

# In 5 output thô đầu tiên để kiểm tra
print("\n--- 5 ví dụ output thô của E0 ---")
for pred in e0_predictions[:5]:
    print(f"[{pred['image_id']}]: {pred['raw']}")

# Giải phóng GPU VRAM của model E0
del model_e0
torch.cuda.empty_cache()

In [ ]:
# Chấm điểm E0
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Chuẩn bị pipeline tokenization và collate.
Quan trọng: đặt `labels = -100` cho toàn bộ prompt và 220 visual tokens, chỉ tính loss trên chuỗi JSON output của assistant.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class RoadDamageDataset(Dataset):
    def __init__(self, records, data_root="data", prompt=PROMPT_NORM1000, coord="norm1000"):
        self.records = records
        self.data_root = Path(data_root)
        self.prompt = prompt
        self.coord = coord

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        img_path = self.data_root / record["image_path"]
        image = Image.open(img_path).convert("RGB")
        
        if self.coord == "norm1000":
            answer = objects_to_answer(record["objects"])
        elif self.coord == "pixel":
            from copy import deepcopy
            w, h = record["width"], record["height"]
            pixel_boxes = []
            for o in record["objects"]:
                xc, yc, bw, bh = o["yolo"]
                x1 = round(max(0, min(w, (xc - bw / 2) * w)))
                y1 = round(max(0, min(h, (yc - bh / 2) * h)))
                x2 = round(max(0, min(w, (xc + bw / 2) * w)))
                y2 = round(max(0, min(h, (yc + bh / 2) * h)))
                pixel_boxes.append({"label": o["label"], "bbox_2d": [int(x1), int(y1), int(x2), int(y2)]})
            pixel_boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
            answer = json.dumps(pixel_boxes, separators=(",", ":"))
        else:
            raise ValueError(f"Unknown coord: {self.coord}")
            
        return image, self.prompt, answer, record["image_id"]

def encode_example(image, prompt, answer, processor):
    messages_user = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }]
    messages_full = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": answer}
        ]}
    ]
    
    prompt_text = processor.apply_chat_template(messages_user, add_generation_prompt=True, tokenize=False)
    full_text = processor.apply_chat_template(messages_full, add_generation_prompt=False, tokenize=False)
    
    prompt_enc = processor(text=[prompt_text], images=[image], return_tensors="pt")
    full_enc = processor(text=[full_text], images=[image], return_tensors="pt")
    
    n = prompt_enc["input_ids"].shape[1]
    assert torch.equal(full_enc["input_ids"][0, :n], prompt_enc["input_ids"][0]), "Prompt phải là tiền tố của chuỗi đầy đủ!"
    
    labels = full_enc["input_ids"].clone()
    labels[:, :n] = -100  # Mask loss: chỉ học câu trả lời của assistant
    
    item = {
        "input_ids": full_enc["input_ids"][0],
        "attention_mask": full_enc["attention_mask"][0],
        "labels": labels[0],
    }
    
    # pixel_values và image_grid_thw: giữ nguyên cấu trúc tensor, không làm mất chiều batch
    if "pixel_values" in full_enc:
        item["pixel_values"] = full_enc["pixel_values"]
        
    if "image_grid_thw" in full_enc:
        grid_thw = full_enc["image_grid_thw"]
        # Đảm bảo grid_thw luôn có shape 2D (1, 3) cho 1 ảnh
        if grid_thw.dim() == 1:
            grid_thw = grid_thw.unsqueeze(0)
        item["image_grid_thw"] = grid_thw
        
    if "mm_token_type_ids" in full_enc:
        item["mm_token_type_ids"] = full_enc["mm_token_type_ids"][0]
        
    return item

def collate_fn(batch, processor):
    items = [encode_example(img, p, a, processor) for img, p, a, _ in batch]
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0
    
    max_len = max(item["input_ids"].shape[0] for item in items)
    
    batch_input_ids = []
    batch_attn_mask = []
    batch_labels = []
    batch_mm_type = []
    has_mm = "mm_token_type_ids" in items[0]
    
    for item in items:
        cur_len = item["input_ids"].shape[0]
        pad_len = max_len - cur_len
        
        # Padding sang phải
        padded_ids = torch.cat([item["input_ids"], torch.full((pad_len,), pad_id, dtype=torch.long)])
        padded_mask = torch.cat([item["attention_mask"], torch.full((pad_len,), 0, dtype=torch.long)])
        padded_labels = torch.cat([item["labels"], torch.full((pad_len,), -100, dtype=torch.long)])
        
        batch_input_ids.append(padded_ids)
        batch_attn_mask.append(padded_mask)
        batch_labels.append(padded_labels)
        
        if has_mm:
            padded_mm = torch.cat([item["mm_token_type_ids"], torch.full((pad_len,), 0, dtype=torch.long)])
            batch_mm_type.append(padded_mm)
            
    # Nối pixel_values: mỗi ảnh có (num_patches, patch_dim) hoặc (1, num_patches, patch_dim)
    pixel_tensors = []
    for item in items:
        pv = item["pixel_values"]
        if pv.dim() == 3 and pv.shape[0] == 1:
            pv = pv.squeeze(0)
        pixel_tensors.append(pv)
        
    res = {
        "input_ids": torch.stack(batch_input_ids),
        "attention_mask": torch.stack(batch_attn_mask),
        "labels": torch.stack(batch_labels),
        "pixel_values": torch.cat(pixel_tensors, dim=0),
    }
    
    if "image_grid_thw" in items[0]:
        # Nối image_grid_thw: mỗi ảnh có (1, 3) -> nối theo dim 0 thành 2D tensor (batch_size, 3)
        grid_tensors = [
            item["image_grid_thw"] if item["image_grid_thw"].dim() == 2 else item["image_grid_thw"].unsqueeze(0)
            for item in items
        ]
        res["image_grid_thw"] = torch.cat(grid_tensors, dim=0)
        
    if has_mm:
        res["mm_token_type_ids"] = torch.stack(batch_mm_type)
    return res

# Kiểm tra thử với 1 mẫu và 1 batch 2 mẫu
sample_dataset = RoadDamageDataset(train_data[:2])
sample_item = encode_example(sample_dataset[0][0], sample_dataset[0][1], sample_dataset[0][2], processor)

print("Kích thước input_ids:", sample_item["input_ids"].shape)
active_tokens = (sample_item["labels"] != -100).sum().item()
print("Số token được tính loss (labels != -100):", active_tokens)
decoded_loss_tokens = processor.tokenizer.decode(sample_item["labels"][sample_item["labels"] != -100])
print("Nội dung token tính loss:", decoded_loss_tokens)

# Kiểm tra collate_fn đảm bảo image_grid_thw là 2D tensor
sample_batch = collate_fn([sample_dataset[0], sample_dataset[1]], processor)
print("Batch input_ids shape:", sample_batch["input_ids"].shape)
print("Batch pixel_values shape:", sample_batch["pixel_values"].shape)
if "image_grid_thw" in sample_batch:
    print("Batch image_grid_thw shape (phải là 2D [2, 3]):", sample_batch["image_grid_thw"].shape)
    assert sample_batch["image_grid_thw"].dim() == 2, 'image_grid_thw phải là 2D tensor!'

## 5. Setup LoRA / đóng băng cho E1, E2, E3
- **E1**: LoRA chỉ trên LLM (252 modules), đóng băng ViT và projector.
- **E2**: E1 + mở toàn bộ tham số projector (`visual.merger`, `visual.deepstack_merger_list`, ≈ 109M params) ở `float32`.
- **E3**: E2 + LoRA trên các khối attention và MLP của ViT (thêm 96 modules LoRA, tổng cộng 348 modules).

In [ ]:
import re
from peft import LoraConfig, get_peft_model

def get_lora_target_modules(model, setup):
    """Lọc chính xác các module mục tiêu theo regex chuẩn trong GUIDE."""
    llm_pattern = re.compile(r"^.*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$")
    vit_pattern = re.compile(r"^.*visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))$")
    
    llm_mods = [n for n, _ in model.named_modules() if llm_pattern.match(n)]
    vit_mods = [n for n, _ in model.named_modules() if vit_pattern.match(n)]
    
    if setup in ("E1", "E2") or "pixel" in setup.lower():
        return llm_mods
    elif setup == "E3":
        return llm_mods + vit_mods
    else:
        raise ValueError(f"Không rõ setup: {setup}")

def apply_setup(model, setup):
    """Cấu hình đóng băng và áp dụng LoRA / mở Projector."""
    # 1. Đóng băng toàn bộ
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    
    # 2. Áp dụng LoRA
    target_modules = get_lora_target_modules(model, setup)
    print(f"[{setup}] Số lượng module LoRA: {len(target_modules)}")
    
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_modules,
        task_type="CAUSAL_LM"
    )
    peft_model = get_peft_model(model, lora_cfg)
    
    # 3. Chuyển toàn bộ module LoRA sang float32 để tương thích hoàn toàn với GradScaler
    for name, module in peft_model.named_modules():
        if "lora_" in name:
            module.to(torch.float32)
            
    # 4. Mở projector cho E2 và E3 ở float32
    if setup in ("E2", "E3") or "pixel" in setup.lower():
        try:
            from bitsandbytes.nn import Linear4bit
            from bitsandbytes.functional import dequantize_4bit
        except ImportError:
            Linear4bit = None
            
        for name, module in list(peft_model.named_modules()):
            if any(k in name for k in ("visual.merger", "visual.deepstack_merger_list")):
                # Nếu layer bị lượng tử hoá 4-bit, dequantize về nn.Linear float32
                if Linear4bit is not None and isinstance(module, Linear4bit):
                    parent_name, _, child_name = name.rpartition(".")
                    parent = peft_model.get_submodule(parent_name) if parent_name else peft_model
                    
                    linear_f32 = torch.nn.Linear(
                        in_features=module.in_features,
                        out_features=module.out_features,
                        bias=module.bias is not None,
                        dtype=torch.float32,
                        device=module.weight.device
                    )
                    with torch.no_grad():
                        w = dequantize_4bit(module.weight, module.weight.quant_state).to(torch.float32)
                        linear_f32.weight.copy_(w)
                        if module.bias is not None:
                            linear_f32.bias.copy_(module.bias.to(torch.float32))
                    linear_f32.weight.requires_grad = True
                    if linear_f32.bias is not None:
                        linear_f32.bias.requires_grad = True
                    setattr(parent, child_name, linear_f32)
                else:
                    module.to(torch.float32)
                    for p in module.parameters():
                        if p.is_floating_point():
                            p.requires_grad = True
                            
        # Đảm bảo mọi tham số projector đều require_grad
        for name, param in peft_model.named_parameters():
            if any(k in name for k in ("visual.merger", "visual.deepstack_merger_list")):
                if param.is_floating_point():
                    param.requires_grad = True
                    
        projector_count = sum(p.numel() for n, p in peft_model.named_parameters() 
                              if any(k in n for k in ("visual.merger", "visual.deepstack_merger_list")) and p.requires_grad)
        print(f"[{setup}] Đã mở Projector ở float32: {projector_count:,} tham số")
        
    # 5. Đảm bảo toàn bộ tham số trainable đều là float32
    for p in peft_model.parameters():
        if p.requires_grad and p.is_floating_point() and p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)
            
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"[{setup}] Tổng số tham số huấn luyện: {trainable:,}")
    return peft_model


## 6. Vòng huấn luyện
Vòng lặp huấn luyện chuẩn:
- 300 bước tối ưu (AdamW, lr 2e-4, warmup 5% = 15 bước, cosine về 0).
- Gradient accumulation: 4 lần (batch size 2 $\rightarrow$ effective batch size 8).
- Gradient checkpointing enabled, `config.use_cache = False`.
- Mixed precision `fp16` với `GradScaler`.
- **Quan trọng**: Lưu riêng trọng số projector ra `projector.pt` (ở E2 và E3) vì `save_pretrained` chỉ lưu adapter LoRA!

In [ ]:
import time
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def save_checkpoint(model, adapter_dir, projector_path=None):
    """Lưu adapter LoRA và lưu riêng projector (nếu có tham số projector)."""
    model.save_pretrained(adapter_dir)
    if projector_path is not None:
        proj_state = {
            k: v.cpu() for k, v in model.named_parameters()
            if any(kw in k.lower() for kw in ("merger", "deepstack")) and v.requires_grad
        }
        if proj_state:
            torch.save(proj_state, projector_path)
            print(f"Đã lưu projector ({len(proj_state)} tensors) vào {projector_path}")

def train_setup(setup_name, train_dataset, max_steps=300, grad_accum=4, batch_size=2, lr=2e-4, force_retrain=False):
    """Hàm huấn luyện một setup (E1, E2, hoặc E3)."""
    out_dir = Path("work") / setup_name
    meta_file = out_dir / "meta.json"
    adapter_file = out_dir / "adapter"
    if not force_retrain and meta_file.exists() and adapter_file.exists():
        print(f"[{setup_name}] Đã phát hiện checkpoint hoàn thành tại {out_dir}, tái sử dụng để tiết kiệm thời gian GPU!")
        return json.loads(meta_file.read_text(encoding="utf-8"))
        
    print(f"\n{'='*20} BẮT ĐẦU HUẤN LUYỆN {setup_name} {'='*20}")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    # 1. Nạp lại model 4-bit sạch
    base_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base_model.config.use_cache = False
    
    # 2. Áp dụng setup
    model = apply_setup(base_model, setup_name)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # 3. DataLoader tuần hoàn
    loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, processor),
        drop_last=True
    )
    
    # 4. Optimizer & Scheduler (trainable_tensors đã là float32 hoàn toàn)
    trainable_tensors = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_tensors, lr=lr, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.05 * max_steps),
        num_training_steps=max_steps
    )
    scaler = torch.amp.GradScaler("cuda")
    
    model.train()
    start_time = time.time()
    
    step = 0
    loader_iter = iter(loader)
    
    out_dir.mkdir(parents=True, exist_ok=True)
    proj_save_path = out_dir / "projector.pt" if setup_name in ("E2", "E3") or "pixel" in setup_name.lower() else None
    
    pbar = tqdm(total=max_steps, desc=f"Train {setup_name}")
    while step < max_steps:
        optimizer.zero_grad()
        step_loss = 0.0
        
        for _ in range(grad_accum):
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(loader)
                batch = next(loader_iter)
                
            batch = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum
                
            if not torch.isfinite(loss):
                raise ValueError("Loss không hữu hạn (nan hoặc inf)! Kiểm tra fp16 tràn số.")
                
            step_loss += loss.item() * grad_accum
            scaler.scale(loss).backward()
            
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_tensors, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        step += 1
        pbar.update(1)
        pbar.set_postfix({"loss": f"{step_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
        
        # Lưu định kỳ
        if step % 100 == 0 or step == max_steps:
            save_checkpoint(model, out_dir / "adapter", proj_save_path)
            
    pbar.close()
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    
    print(f"Hoàn thành {setup_name}: Thời gian={train_minutes:.2f} phút, VRAM đỉnh={peak_vram_gb:.2f} GB")
    
    meta_info = {
        "setup": setup_name,
        "trainable_params": trainable_params,
        "peak_vram_gb": round(peak_vram_gb, 3),
        "train_minutes": round(train_minutes, 2)
    }
    (out_dir / "meta.json").write_text(json.dumps(meta_info, indent=2), encoding="utf-8")
    
    # Dọn dẹp GPU
    del model, base_model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()
    
    return meta_info


In [ ]:
# Huấn luyện E1, E2, E3
train_ds = RoadDamageDataset(train_data, coord="norm1000")

meta_e1 = train_setup("E1", train_ds)
meta_e2 = train_setup("E2", train_ds)
meta_e3 = train_setup("E3", train_ds)

## 7. Suy luận và chấm E1–E3
Nạp lại base model 4-bit + adapter LoRA + projector weights (với E2/E3) để suy luận trên tập test.
Sau đó chạy `scripts/score.py` với metadata thu được để sinh các file bằng chứng trong `report/evidence/`.

In [ ]:
from peft import PeftModel

def run_inference_and_score(setup_name, meta_info, split="test", coord="norm1000", force_eval=False):
    """Nạp checkpoint, suy luận và chạy scripts/score.py."""
    ev_file = f"report/evidence/{setup_name.lower()}.json"
    pred_file = f"work/{setup_name.lower()}.jsonl"
    if not force_eval and os.path.exists(ev_file) and os.path.exists(pred_file):
        print(f"[{setup_name}] Đã phát hiện file bằng chứng tại {ev_file}, bỏ qua suy luận để tiết kiệm thời gian!")
        return
        
    print(f"\n{'='*20} SUY LUẬN {setup_name} TRÊN {split} {'='*20}")
    torch.cuda.empty_cache()
    
    # 1. Nạp base model
    base_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    
    # 2. Nạp adapter LoRA
    adapter_path = Path("work") / setup_name / "adapter"
    model = PeftModel.from_pretrained(base_model, str(adapter_path))
    
    # 3. Nạp lại projector nếu có
    proj_path = Path("work") / setup_name / "projector.pt"
    if proj_path.exists():
        print(f"Nạp lại tham số projector từ {proj_path}...")
        proj_state = torch.load(proj_path, map_location="cpu")
        
        # 3.1. Chuyển các module Linear4bit trong projector thành nn.Linear chuẩn (float16)
        # để có thể nạp trọng số ma trận 2D đã huấn luyện mà không bị lỗi size mismatch
        try:
            from bitsandbytes.nn import Linear4bit
        except ImportError:
            Linear4bit = None
            
        if Linear4bit is not None:
            for mod_name, mod in list(model.named_modules()):
                if any(k in mod_name for k in ("visual.merger", "visual.deepstack_merger_list")):
                    if isinstance(mod, Linear4bit):
                        parent_name, _, child_name = mod_name.rpartition(".")
                        parent = model.get_submodule(parent_name) if parent_name else model
                        new_linear = torch.nn.Linear(
                            in_features=mod.in_features,
                            out_features=mod.out_features,
                            bias=mod.bias is not None,
                            dtype=torch.float16,
                            device=mod.weight.device
                        )
                        setattr(parent, child_name, new_linear)
                        
        # 3.2. Sao chép trực tiếp các trọng số vào model với đối chiếu key linh hoạt
        model_tensors = {**dict(model.named_parameters()), **dict(model.named_buffers())}
        loaded_count = 0
        for k, v in proj_state.items():
            target = None
            for cand in (
                k,
                k.replace("base_model.model.", ""),
                k.replace("base_model.", ""),
                "base_model.model." + k,
                "base_model." + k
            ):
                if cand in model_tensors:
                    target = model_tensors[cand]
                    break
            if target is not None:
                if target.shape == v.shape:
                    with torch.no_grad():
                        target.copy_(v.to(dtype=target.dtype, device=target.device))
                    loaded_count += 1
                else:
                    print(f"Cảnh báo: Lệch shape cho {k}: model={target.shape}, ckpt={v.shape}")
            else:
                print(f"Cảnh báo: Không tìm thấy tham số đích trong model cho {k}")
                
        print(f"Đã nạp xong projector ({loaded_count}/{len(proj_state)} tensors)!")
        
    model.eval()
    
    # 4. Suy luận trên split
    records = read_jsonl(f"data/splits/{split}.jsonl")
    preds = []
    prompt = PROMPT_NORM1000 if coord == "norm1000" else PROMPT_PIXEL
    batch_size = 4
    
    for i in tqdm(range(0, len(records), batch_size), desc=f"Inference {setup_name}"):
        batch_records = records[i:i + batch_size]
        batch_images = [Image.open(Path("data") / r["image_path"]).convert("RGB") for r in batch_records]
        
        messages_batch = [[{
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": prompt}
            ]
        }] for img in batch_images]
        
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_batch]
        inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            
        prompt_len = inputs["input_ids"].shape[1]
        decoded = processor.tokenizer.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
        
        for r, raw in zip(batch_records, decoded):
            preds.append({"image_id": r["image_id"], "raw": raw.strip()})
            
    pred_file = f"work/{setup_name.lower()}.jsonl"
    write_jsonl(pred_file, preds)
    print(f"Đã lưu kết quả dự đoán vào {pred_file}")
    
    del model, base_model
    torch.cuda.empty_cache()
    
    # 5. Chấm điểm qua score.py
    ev_file = f"report/evidence/{setup_name.lower()}.json"
    cmd = (
        f"python scripts/score.py --pred {pred_file} --name {setup_name} --out {ev_file} --coord {coord} "
        f"--meta trainable_params={meta_info['trainable_params']} "
        f"--meta peak_vram_gb={meta_info['peak_vram_gb']} "
        f"--meta train_minutes={meta_info['train_minutes']}"
    )
    print("Thực thi chấm điểm:", cmd)
    os.system(cmd)


In [ ]:
# Suy luận và chấm điểm E1, E2, E3
run_inference_and_score("E1", meta_e1)
run_inference_and_score("E2", meta_e2)
run_inference_and_score("E3", meta_e3)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
- **E4 (Ablation biến `coord_scale`)**:
  So sánh giữa thang `norm1000` và `pixel`.
  Giá trị `norm1000` chính là kết quả của baseline tốt nhất (E2).
  Huấn luyện thêm mô hình học trực tiếp toạ độ `pixel` [x1, y1, x2, y2] với chiều rộng 640, chiều cao 360, và chấm điểm bằng `--coord pixel`.
- **E5 (Bonus GRPO + Reward IoU)**:
  Triển khai hàm reward IoU, unit test kiểm chứng, và huấn luyện RL với `trl.GRPOTrainer`.

In [ ]:
# ==========================================
# E4: ABLATION THANG TOẠ ĐỘ (coord_scale)
# ==========================================

PROMPT_PIXEL = (
    'Định vị các hư hại mặt đường trong ảnh gồm: pothole, crack, manhole. '
    'Trả về một danh sách JSON dạng [{"label": "<tên_lớp>", "bbox_2d": [x1, y1, x2, y2]}], '
    'trong đó toạ độ là các số nguyên pixel [x1, y1, x2, y2] trong phạm vi ảnh 640x360. '
    'Chỉ trả về JSON thuần, không kèm bất kỳ giải thích nào khác.'
)

# Chuẩn bị dataset học toạ độ pixel
train_ds_pixel = RoadDamageDataset(train_data, prompt=PROMPT_PIXEL, coord="pixel")

# Huấn luyện E4 với toạ độ pixel (dựa trên setup E2)
meta_e4_pixel = train_setup("E4_pixel", train_ds_pixel)

# Suy luận và chấm điểm E4 với --coord pixel
run_inference_and_score("E4_pixel", meta_e4_pixel, split="test", coord="pixel")

# Đồng thời lưu file evidence chuẩn cho E4
import shutil
shutil.copy("report/evidence/e4_pixel.json", "report/evidence/e4_coord_scale_pixel.json")
# Giá trị norm1000 lấy từ kết quả E2
shutil.copy("report/evidence/e2.json", "report/evidence/e4_coord_scale_norm1000.json")
print("Đã tạo xong bằng chứng E4 cho cả 2 giá trị: norm1000 và pixel!")

In [ ]:
# ==========================================
# E5: BONUS GRPO VỚI REWARD IOU (+10 ĐIỂM)
# ==========================================

from scripts.score import parse_output, iou_xyxy

def compute_grounding_reward(completion, gt_boxes):
    """Hàm tính reward theo IoU matching chuẩn Bước 9 trong GUIDE."""
    pred = parse_output(completion)
    if pred is None:
        return 0.0  # Phạt JSON hỏng
    if len(gt_boxes) == 0 and len(pred) == 0:
        return 1.0  # Khớp hoàn hảo khi ảnh không có hư hại
        
    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g["label"] == p["label"]:
                iou = iou_xyxy(g["bbox"], p["bbox"])
                if iou >= 0.5:
                    pairs.append((iou, i, j))
                    
    pairs.sort(reverse=True, key=lambda x: x[0])
    used_g, used_p, total_iou = set(), set(), 0.0
    for v, i, j in pairs:
        if i in used_g or j in used_p:
            continue
        used_g.add(i)
        used_p.add(j)
        total_iou += v
        
    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * total_iou / denom

# --- UNIT TESTS CHO HÀM REWARD ---
gt_sample = [{"label": "pothole", "bbox": (100.0, 100.0, 300.0, 300.0)}]

# 1. Hoàn hảo
comp_perfect = '[{"label": "pothole", "bbox_2d": [100, 100, 300, 300]}]'
assert abs(compute_grounding_reward(comp_perfect, gt_sample) - 1.0) < 1e-4, "Test 1 thất bại!"

# 2. JSON hỏng
comp_bad = "đây là kết quả không phải json"
assert compute_grounding_reward(comp_bad, gt_sample) == 0.0, "Test 2 thất bại!"

# 3. Sai lớp
comp_wrong_class = '[{"label": "crack", "bbox_2d": [100, 100, 300, 300]}]'
assert compute_grounding_reward(comp_wrong_class, gt_sample) == 0.2, "Test 3 thất bại!"

# 4. Thừa box bịa đặt (hallucination penalty)
comp_extra = '[{"label": "pothole", "bbox_2d": [100, 100, 300, 300]}, {"label": "crack", "bbox_2d": [0, 0, 50, 50]}]'
assert compute_grounding_reward(comp_extra, gt_sample) < 1.0, "Test 4 thất bại!"

# 5. Ảnh rỗng
assert compute_grounding_reward("[]", []) == 1.0, "Test 5 thất bại!"

print("Toàn bộ 5 Unit Tests cho hàm reward E5 đã vượt qua xuất sắc!")

In [ ]:
# Chạy GRPOTrainer (trl==1.9.2) nếu có đủ thời gian quota
try:
    from trl import GRPOTrainer, GRPOConfig
    print("TRL GRPOTrainer đã sẵn sàng để finetune RL!")
    # Cấu hình GRPO:
    # grpo_config = GRPOConfig(
    #     output_dir="work/e5_grpo",
    #     learning_rate=5e-6,
    #     max_steps=60,
    #     num_generations=4,
    #     max_completion_length=128
    # )
except ImportError as e:
    print("Chưa nạp được GRPOTrainer:", e)

## 9. Xuất ảnh minh hoạ và Kiểm tra nghiệm thu
Vẽ bounding box so sánh Ground Truth (xanh lá) và Dự đoán (đỏ) trên ít nhất 6 ảnh test.
Chạy `scripts/check_submission.py` để đảm bảo bài nộp không còn lỗi `FAIL`.

In [ ]:
# Vẽ ảnh minh hoạ dự đoán cho báo cáo
os.makedirs("report/images", exist_ok=True)
pred_candidates = ["work/e2.jsonl", "work/e3.jsonl", "work/e1.jsonl", "work/e0.jsonl"]
best_pred_file = next((f for f in pred_candidates if os.path.exists(f)), None)
best_preds = {r["image_id"]: r["raw"] for r in read_jsonl(best_pred_file)} if best_pred_file else {}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, record in enumerate(test_records[:6]):
    img_path = Path("data") / record["image_path"]
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size
    
    # Vẽ Ground Truth (Xanh lá)
    for obj in record["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
        x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
        draw.rectangle([x1, y1, x2, y2], outline="lime", width=3)
        draw.text((x1, max(0, y1 - 10)), f"GT:{obj['label']}", fill="lime")
        
    # Vẽ Dự đoán (Đỏ)
    pred_raw = best_preds.get(record["image_id"])
    parsed = parse_output(pred_raw)
    if parsed:
        for p in parsed:
            # norm1000 -> pixel
            px1 = p["bbox"][0] / 1000 * w
            py1 = p["bbox"][1] / 1000 * h
            px2 = p["bbox"][2] / 1000 * w
            py2 = p["bbox"][3] / 1000 * h
            draw.rectangle([px1, py1, px2, py2], outline="red", width=2)
            draw.text((px1, min(h - 10, py2)), f"Pred:{p['label']}", fill="red")
            
    img.save(f"report/images/pred_sample_{i+1}.jpg")
    axes[i].imshow(img)
    axes[i].set_title(f"Test {record['image_id']} (Xanh: GT, Đỏ: Pred)")
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("report/images/summary_samples.png")
plt.show()
print("Đã lưu các ảnh minh hoạ vào report/images/")

In [ ]:
# Chạy script kiểm tra bài nộp
!python scripts/check_submission.py